In [38]:
%load_ext autoreload
%autoreload 2
import pickle
import numpy as np
import pandas as pd
import wsb_utils as wu
import model_utils as mu

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=43373, as_frame=True, parser="auto").frame
print(df.shape)
df.columns

(53187, 7)


Index(['title', 'score', 'url', 'comms_num', 'created', 'body', 'timestamp'],
      dtype='str')

In [40]:
url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt"
sym = pd.read_csv(url, sep="|")
valid_tickers = set(sym["Symbol"].dropna().astype(str))
print(len(valid_tickers), "symbols")

13285 symbols


In [41]:
start = wu.first_full_day(df, date_col="created")
print("Data starts properly on:", start.date())

days = wu.to_trading_day(df["created"])
df_full = df[days.values >= start]
print(len(df_full), "posts kept of", len(df))

mentions = wu.daily_mentions(df_full, valid_tickers, date_col="created")
spikes = wu.find_spikes(mentions, ratio=5, min_mentions=5)
print(len(spikes), "spikes")
spikes.head(20)

Data starts properly on: 2021-01-28
53186 posts kept of 53187
121 spikes


,date,ticker,mentions,avg_before
0,2021-02-11,ACB,5,0.857143
1,2021-02-11,TLRY,21,2.500000
2,2021-02-12,TSM,6,0.500000
3,2021-02-15,IBM,5,0.500000
4,2021-02-15,INTC,7,0.428571
5,2021-02-15,MVIS,6,0.428571
6,2021-02-15,NVDA,7,0.571429
7,2021-02-18,RIOT,5,0.571429
8,2021-02-19,PLTR,37,7.142857
9,2021-02-22,AAPL,6,1.142857


In [42]:
post_days = wu.to_trading_day(df_full["created"])
print("Posts: ", post_days.min().date(), "→", post_days.max().date())
print("Spikes:", spikes["date"].min().date(), "→", spikes["date"].max().date())

print("\nPosts per month:")
print(post_days.dt.to_period("M").value_counts().sort_index())

print("\nTop 40 most-mentioned symbols:")
print(mentions.groupby("ticker")["mentions"].sum().sort_values(ascending=False).head(40))

print("\nSpikes per stock:")
print(spikes["ticker"].value_counts().head(30))

print("\nSpikes per month:")
print(spikes["date"].dt.to_period("M").value_counts().sort_index())

Posts:  2021-01-28 → 2021-08-16
Spikes: 2021-02-11 → 2021-08-16

Posts per month:
created
2021-01    16829
2021-02    19404
2021-03     7740
2021-04     1989
2021-05     2261
2021-06     2748
2021-07     1364
2021-08      851
Freq: M, Name: count, dtype: int64

Top 40 most-mentioned symbols:
ticker
GME     12055
AMC      4485
BB       1866
NOK      1364
PLTR      752
RKT       715
TSLA      551
UWMC      382
AMD       348
SNDL      338
CLOV      308
SPCE      274
AAPL      213
NIO       204
SLV       200
TLRY      175
CLNE      169
CRSR      166
CLF       164
AAL       156
MVIS      149
AMZN      147
WKHS      144
GM        143
ASO       129
BABA      118
DKNG      107
MSFT       98
NVDA       97
FB         96
SOFI       89
IBKR       84
JPM        83
AG         82
MUST       78
GE         76
KOSS       75
ARKK       70
WEEK       70
CASH       69
Name: mentions, dtype: int64

Spikes per stock:
ticker
PLTR    9
RKT     9
UWMC    8
MVIS    6
BB      6
AMD     6
ASO     5
AMC     5
AAPL 

In [43]:
for s in ["ASO", "GM", "IP", "AG"]:
    titles = df_full[df_full["title"].str.contains(rf"\b{s}\b", na=False)]["title"].head(5).tolist()
    print(s, titles, "\n")

ASO ['ASO STOCKS', 'ASO is free money.', '(ASO) Academy Sports and Outdoors Next RKT similar fundamentals', 'ASO - DD - First timer - Spreading some knowledge', "Half my account into ASO calls today! Scared money don't make money! "] 



GM ['BUY GM', 'What is we made a thread about GM.E not GME.', 'Stock Market News for Today  GM / SQ / UBER  The Short Squeeze  other Stock Market News [01-29]', 'BREAKING: Robinhood now limiting trading in 50+ stocks, including GM, Beyond Meat, and Starbucks', 'Ford Motor Company, will they respond to GM with an all electric model lineup?'] 

IP ['Complete DD on HYLN. No battery patent, no IP, no product. Nikola 2.0?', 'Yolo on Pure options 1TDE. If GME IS 500 on 2-26 I will put the WSB logo on ice cream cups packaging I mass manufacture (can someone give me some advice on the IP?)', 'BUY LABU/XBI FAAAAAAAST - MERKEL OPPOSES IP WAVER', 'Potential waiving of IP rights for Covid-19 Vaccines'] 

AG ["Once you're done with GME - AG and SLV, the gentleman's short squeeze, driven by macro fundamentals", 'IB just halted AG trading', 'Is The Reddit Rebellion About To Descend On The Precious Metals Market? SLV AG', 'AG', 'AG +30, PAAS + 10, SLV +4.60, PSLV +4.68, ... Is it happening ?'] 



In [44]:
results = []
for ratio in [3, 4, 5]:
    for min_m in [5, 8, 10]:
        n = len(wu.find_spikes(mentions, ratio=ratio, min_mentions=min_m))
        results.append({"ratio": ratio, "min_mentions": min_m, "spikes": n})
pd.DataFrame(results).pivot(index="ratio", columns="min_mentions", values="spikes")

min_mentions,5,8,10
ratio,,,
3,183,103,78
4,144,77,58
5,121,64,46


In [45]:
price_start = spikes["date"].min() - pd.Timedelta(days=60)   # history for the clues
price_end   = spikes["date"].max() + pd.Timedelta(days=15)   # future for the label

tickers = sorted(spikes["ticker"].unique()) + ["SPY"]
prices = mu.get_prices(tickers, price_start, price_end)

print(len(prices), "of", len(tickers), "tickers downloaded")
print("Missing:", sorted(set(tickers) - set(prices)))

pickle.dump(prices, open("prices.pkl", "wb"))

48 of 49 tickers downloaded
Missing: ['HOOD']


In [46]:
data = mu.build_dataset(spikes, prices)
print(len(data), "spikes with enough price history")

data = data.sort_values(["ticker", "date"])
gap = data.groupby("ticker")["date"].diff().dt.days
data = data[gap.isna() | (gap > 7)].dropna(subset=mu.FEATURES)
data = data.sort_values("date").reset_index(drop=True)
print(len(data), "examples after removing repeats")

116 spikes with enough price history
80 examples after removing repeats


In [47]:
for t in [-0.10, -0.15, -0.20]:
    hit = data["min_future_ret"] <= t
    print(f"fall of {abs(t):.0%}+: {hit.sum()} drops ({hit.mean():.0%} of spikes)")

DROP = -0.10
data["crashed"] = (data["min_future_ret"] <= DROP).astype(int)
print("\nSharp-drop rate:", round(data["crashed"].mean(), 3), "| drops:", data["crashed"].sum())

fall of 10%+: 28 drops (35% of spikes)
fall of 15%+: 14 drops (18% of spikes)
fall of 20%+: 7 drops (9% of spikes)

Sharp-drop rate: 0.35 | drops: 28


In [48]:
cutoff = data["date"].iloc[int(len(data) * 0.7)]

train = data[data["date"] < cutoff - pd.Timedelta(days=7)]
test  = data[data["date"] >= cutoff]

print("Cutoff:", cutoff.date())
print(f"Train: {len(train)} spikes, {train['crashed'].sum()} drops  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test:  {len(test)} spikes, {test['crashed'].sum()} drops  ({test['date'].min().date()} → {test['date'].max().date()})")

X_train, y_train = train[mu.FEATURES], train["crashed"]
X_test,  y_test  = test[mu.FEATURES],  test["crashed"]

Cutoff: 2021-06-14
Train: 46 spikes, 18 drops  (2021-02-11 → 2021-05-28)
Test:  24 spikes, 8 drops  (2021-06-14 → 2021-08-16)


In [49]:
from sklearn.metrics import precision_score, recall_score, average_precision_score

def report(name, y, pred):
    pred = np.asarray(pred).astype(int)
    print(f"{name:28s} precision {precision_score(y, pred, zero_division=0):.2f}   "
          f"recall {recall_score(y, pred, zero_division=0):.2f}   warnings {pred.sum()}/{len(y)}")

report("Always warn",            y_test, np.ones(len(y_test)))
report("Rule: volume 3x normal", y_test, test["volume_ratio"] >= 3)
report("Rule: already up 50%+",  y_test, test["ret_5d"] >= 0.5)

Always warn                  precision 0.33   recall 1.00   warnings 24/24
Rule: volume 3x normal       precision 0.71   recall 0.62   warnings 7/24
Rule: already up 50%+        precision 1.00   recall 0.25   warnings 2/24


In [50]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                             class_weight="balanced", random_state=42),
}

probas = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probas[name] = model.predict_proba(X_test)[:, 1]
    report(name, y_test, probas[name] >= 0.5)
    print(f"{'':28s} PR-AUC {average_precision_score(y_test, probas[name]):.2f}  "
          f"(random guessing ≈ {y_test.mean():.2f})")

Logistic regression          precision 0.60   recall 0.38   warnings 5/24
                             PR-AUC 0.73  (random guessing ≈ 0.33)
Random forest                precision 0.78   recall 0.88   warnings 9/24
                             PR-AUC 0.94  (random guessing ≈ 0.33)


In [51]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                             class_weight="balanced", random_state=42),
}

probas = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probas[name] = model.predict_proba(X_test)[:, 1]
    report(name, y_test, probas[name] >= 0.5)
    print(f"{'':28s} PR-AUC {average_precision_score(y_test, probas[name]):.2f}  "
          f"(random guessing ≈ {y_test.mean():.2f})")

Logistic regression          precision 0.60   recall 0.38   warnings 5/24
                             PR-AUC 0.73  (random guessing ≈ 0.33)
Random forest                precision 0.78   recall 0.88   warnings 9/24
                             PR-AUC 0.94  (random guessing ≈ 0.33)


In [52]:
imp = pd.Series(models["Random forest"].feature_importances_, index=mu.FEATURES).sort_values(ascending=False)
print("What the random forest relies on most:")
print(imp.round(3))

coefs = pd.Series(models["Logistic regression"][-1].coef_[0], index=mu.FEATURES).sort_values()
print("\nLogistic regression: + raises risk, - lowers it")
print(coefs.round(2))

vol_cut = train["volatility_20d"].median()
print(f"\nVolatile = daily swings above {vol_cut:.3f} (median of the training spikes)")
report("Rule: volatile stock", y_test, test["volatility_20d"] >= vol_cut)

What the random forest relies on most:
volatility_20d    0.242
spike_ratio       0.168
volume_ratio      0.117
log_price         0.115
ret_5d            0.114
market_ret_5d     0.093
ret_20d           0.086
log_mentions      0.065
dtype: float64

Logistic regression: + raises risk, - lowers it
ret_5d           -0.44
log_price        -0.33
market_ret_5d    -0.27
log_mentions      0.08
volume_ratio      0.09
ret_20d           0.58
spike_ratio       0.73
volatility_20d    0.91
dtype: float64

Volatile = daily swings above 0.037 (median of the training spikes)
Rule: volatile stock         precision 0.86   recall 0.75   warnings 7/24


In [53]:
seeds = []
for seed in range(10):
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                class_weight="balanced", random_state=seed).fit(X_train, y_train)
    p = rf.predict_proba(X_test)[:, 1]
    seeds.append({"precision": precision_score(y_test, p >= 0.5, zero_division=0),
                  "recall": recall_score(y_test, p >= 0.5, zero_division=0),
                  "pr_auc": average_precision_score(y_test, p)})
print("Random forest over 10 random seeds:")
print(pd.DataFrame(seeds).agg(["mean", "min", "max"]).round(2))

rng = np.random.default_rng(0)
fake = []
for i in range(20):
    y_shuffled = rng.permutation(y_train.values)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                class_weight="balanced", random_state=0).fit(X_train, y_shuffled)
    fake.append(average_precision_score(y_test, rf.predict_proba(X_test)[:, 1]))
print(f"\nPR-AUC with SHUFFLED training labels: mean {np.mean(fake):.2f}, max {np.max(fake):.2f}")
print(f"PR-AUC with the real labels:          {average_precision_score(y_test, probas['Random forest']):.2f}")

Random forest over 10 random seeds:
      precision  recall  pr_auc
mean       0.86    0.88    0.93
min        0.78    0.88    0.91
max        0.88    0.88    0.95

PR-AUC with SHUFFLED training labels: mean 0.48, max 0.87
PR-AUC with the real labels:          0.94


In [54]:
table = test[["date", "ticker", "mentions", "ret_5d", "volatility_20d", "min_future_ret", "crashed"]].copy()
table["rf_risk"] = probas["Random forest"].round(2)
table["warned"] = (table["rf_risk"] >= 0.5).astype(int)
table.sort_values("rf_risk", ascending=False)

,date,ticker,mentions,ret_5d,volatility_20d,min_future_ret,crashed,rf_risk,warned
56,2021-06-14,SENS,9,0.328859,0.110305,-0.209596,1,0.75,1
71,2021-07-08,NEGG,8,1.363261,0.376526,-0.449782,1,0.71,1
67,2021-06-28,SPCE,14,0.522488,0.095330,-0.214424,1,0.69,1
59,2021-06-15,WOOF,6,0.073964,0.053651,-0.155844,1,0.62,1
69,2021-06-29,SOFI,20,-0.109906,0.051504,-0.109698,1,0.59,1
75,2021-08-05,AMC,5,-0.121164,0.087767,-0.122650,1,0.58,1
60,2021-06-15,CRSR,6,0.125668,0.034984,-0.125035,1,0.58,1
77,2021-08-12,PLTR,7,0.105240,0.032442,-0.048614,0,0.54,1
74,2021-08-03,AMD,7,0.236515,0.026297,-0.072850,0,0.51,1
62,2021-06-17,ITUB,9,0.004623,0.017243,-0.035276,0,0.49,0


In [55]:
price_only = ["ret_5d", "ret_20d", "volume_ratio", "volatility_20d", "log_price", "market_ret_5d"]
feature_sets = {
    "All clues (Reddit + price)": mu.FEATURES,
    "Price clues only (no Reddit)": price_only,
    "Volatility only": ["volatility_20d"],
}

print(f"Ranking by volatility alone, no model: PR-AUC {average_precision_score(y_test, test['volatility_20d']):.2f}\n")

for name, feats in feature_sets.items():
    aps, recs = [], []
    for seed in range(10):
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                    class_weight="balanced", random_state=seed).fit(train[feats], y_train)
        p = rf.predict_proba(test[feats])[:, 1]
        aps.append(average_precision_score(y_test, p))
        recs.append(recall_score(y_test, p >= 0.5, zero_division=0))
    print(f"{name:30s} PR-AUC {np.mean(aps):.2f} (range {min(aps):.2f}–{max(aps):.2f})   recall {np.mean(recs):.2f}")

Ranking by volatility alone, no model: PR-AUC 0.91

All clues (Reddit + price)     PR-AUC 0.93 (range 0.91–0.95)   recall 0.88
Price clues only (no Reddit)   PR-AUC 0.90 (range 0.88–0.91)   recall 0.79
Volatility only                PR-AUC 0.86 (range 0.86–0.87)   recall 0.50


In [56]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

# Practice guesses on the STUDY pile: each spike is guessed by a forest that didn't study it
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            class_weight="balanced", random_state=42)
practice = cross_val_predict(rf, X_train, y_train, method="predict_proba",
                             cv=StratifiedKFold(5, shuffle=True, random_state=42))[:, 1]

print("Cutoff choices, judged on the STUDY pile only:")
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    report(f"warn if risk >= {t:.0%}", y_train, practice >= t)

Cutoff choices, judged on the STUDY pile only:
warn if risk >= 30%          precision 0.42   recall 1.00   warnings 43/46
warn if risk >= 35%          precision 0.46   recall 0.94   warnings 37/46
warn if risk >= 40%          precision 0.47   recall 0.78   warnings 30/46
warn if risk >= 45%          precision 0.46   recall 0.61   warnings 24/46
warn if risk >= 50%          precision 0.53   recall 0.56   warnings 19/46
warn if risk >= 55%          precision 0.55   recall 0.33   warnings 11/46
warn if risk >= 60%          precision 0.43   recall 0.17   warnings 7/46


In [ ]:
CUTOFF = 0.40   # chosen on the study pile only
final = probas["Random forest"] >= CUTOFF

print(f"FINAL: random forest, warn if risk >= {CUTOFF:.0%}, on the test months")
report("Random forest (final)", y_test, final)
print()
report("Rule: volatile stock", y_test, test["volatility_20d"] >= vol_cut)
report("Always warn",          y_test, np.ones(len(y_test)))